Noteboook que scrappea los registros semanales de las semillas de interés en el rango de enero de 2015 a diciembre de 2024

In [8]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
import re
import time

def scrape_sniim_produccion_continuo():
    url_base = "http://www.economia-sniim.gob.mx/nuevo/Consultas/MercadosNacionales/PreciosDeMercado/Agricolas/ResultadosConsultaFechaGranos.aspx"

    # --- COLOCA TUS IDs REALES AQUÍ ---
    productos_ids = {
        "Maíz blanco": "605",
        "Maíz blanco pozolero": "606",
        "Frijol Pinto": "352",
        "Frijol Negro": "347",
        "Haba": "602"
    }

    destino_id = "100"
    destino_nombre = "DF: Central de Abasto de Iztapalapa DF"

    anios = range(2015, 2025)
    meses = range(1, 13)
    semanas = range(1, 6)

    registros_maestros = []

    session = requests.Session()
    session.headers.update({
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'
    })

    print("Iniciando EXTRACCIÓN MASIVA (Control estricto de valores Nulos)...\n")

    for producto_nombre, producto_id in productos_ids.items():

        print(f"\n--- Procesando: {producto_nombre} ---")

        for anio in anios:
            for mes in meses:
                for semana in semanas:
                    params = {
                        'Semana': semana, 'Mes': mes, 'Anio': anio,
                        'ProductoId': producto_id, 'OrigenId': '-1',
                        'Origen': 'Todos', 'DestinoId': destino_id,
                        'Destino': destino_nombre
                    }

                    datos_encontrados_semana = False
                    es_mes_sin_quinta_semana = False # --- NUEVA BANDERA ---

                    try:
                        response = session.get(url_base, params=params, timeout=10)

                        # --- VERIFICACIÓN DE MESES CORTOS ---
                        if 'NO HAY REGISTROS' in response.text:
                            if semana == 5:
                                es_mes_sin_quinta_semana = True # Es la 5ta semana y no existe, es normal.
                            # Si es la semana 1, 2, 3 o 4 y no hay registros, la bandera sigue False y se registrará como Nulo (Fallo/Faltante real).

                        elif 'Precio Mín' in response.text:
                            soup = BeautifulSoup(response.text, 'html.parser')
                            header_cell = soup.find(lambda tag: tag.name in ['td', 'th'] and 'Precio Mín' in tag.get_text())

                            if header_cell:
                                tabla = header_cell.find_parent('table')
                                todas_las_filas = tabla.find_all('tr')

                                for row in todas_las_filas:
                                    cols = row.find_all('td')
                                    if len(cols) >= 5:
                                        fecha = cols[0].get_text(strip=True)

                                        if not re.match(r'^\d{2}/\d{2}/\d{4}$', fecha):
                                            continue

                                        registros_maestros.append({
                                            'Fecha': fecha,
                                            'Origen': cols[1].get_text(strip=True),
                                            'Precio Mín': cols[2].get_text(strip=True),
                                            'Precio Max': cols[3].get_text(strip=True),
                                            'Precio Frec': cols[4].get_text(strip=True),
                                            'Producto_Buscado': producto_nombre,
                                            'Año': anio,
                                            'Mes': mes,
                                            'Semana': semana
                                        })
                                        datos_encontrados_semana = True

                    except Exception as e:
                        print(f"  [!] Error de conexión: Sem {semana}, Mes {mes}, Año {anio} - {e}")

                    # --- RELLENO DE NULOS INTELIGENTE ---
                    if not datos_encontrados_semana:
                        # Si es la 5ta semana y el mes es corto, simplemente la ignoramos
                        if es_mes_sin_quinta_semana:
                            pass
                        # En cualquier otro caso (error de red o faltante real), inyectamos la fila Nula
                        else:
                            registros_maestros.append({
                                'Fecha': None,
                                'Origen': None,
                                'Precio Mín': None,
                                'Precio Max': None,
                                'Precio Frec': None,
                                'Producto_Buscado': producto_nombre,
                                'Año': anio,
                                'Mes': mes,
                                'Semana': semana
                            })
                            # Opcional: imprimir cuándo ocurre un fallo real
                            # print(f"  [!] Fallo/Faltante real convertido a Null: Sem {semana}, Mes {mes}, Año {anio}")

                    time.sleep(0.1)

    if registros_maestros:
        df_final = pd.DataFrame(registros_maestros)

        for col in ['Precio Mín', 'Precio Max', 'Precio Frec']:
            df_final[col] = pd.to_numeric(df_final[col].astype(str).str.replace(',', ''), errors='coerce')

        df_final.to_csv('precios_historicos_sniim_continuo.csv', index=False, encoding='utf-8-sig')
        print(f"\n¡Extracción finalizada! Se generó el dataset con {len(df_final)} registros.")
        return df_final
    else:
        print("\nNo se extrajeron datos.")
        return pd.DataFrame()

df_final = scrape_sniim_produccion_continuo()

Iniciando EXTRACCIÓN MASIVA (Control estricto de valores Nulos)...


--- Procesando: Maíz blanco ---

--- Procesando: Maíz blanco pozolero ---

--- Procesando: Frijol Pinto ---

--- Procesando: Frijol Negro ---

--- Procesando: Haba ---

¡Extracción finalizada! Se generó el dataset con 2577 registros.


Celda de pruebas para Scrapper

In [4]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
import re  # <--- NUEVA LIBRERÍA PARA EXPRESIONES REGULARES

def prueba_rapida_sniim_estricta():
    url_base = "http://www.economia-sniim.gob.mx/nuevo/Consultas/MercadosNacionales/PreciosDeMercado/Agricolas/ResultadosConsultaFechaGranos.aspx"
    productos_ids = {"Maíz blanco": "605"}
    destino_id = "100"
    destino_nombre = "DF: Central de Abasto de Iztapalapa DF"
    anios = [2015]
    meses = [1]
    semanas = [1, 2, 3]

    datos_completos = []
    session = requests.Session()
    session.headers.update({
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'
    })

    print("Iniciando MODO PRUEBA ESTRICTO...\n")

    for producto_nombre, producto_id in productos_ids.items():
        for anio in anios:
            for mes in meses:
                for semana in semanas:
                    params = {'Semana': semana, 'Mes': mes, 'Anio': anio, 'ProductoId': producto_id, 'OrigenId': '-1', 'Origen': 'Todos', 'DestinoId': destino_id, 'Destino': destino_nombre}

                    try:
                        response = session.get(url_base, params=params, timeout=10)

                        if 'NO HAY REGISTROS' in response.text:
                            continue

                        if 'Precio Mín' in response.text:
                            soup = BeautifulSoup(response.text, 'html.parser')
                            header_cell = soup.find(lambda tag: tag.name in ['td', 'th'] and 'Precio Mín' in tag.get_text())

                            if header_cell:
                                tabla = header_cell.find_parent('table')
                                todas_las_filas = tabla.find_all('tr')

                                extracted_records = []
                                for row in todas_las_filas:
                                    cols = row.find_all('td')
                                    if len(cols) >= 5:
                                        fecha = cols[0].get_text(strip=True)

                                        # --- FILTRO DE TITANIO ---
                                        # Exigimos formato exacto: DD/MM/YYYY
                                        if not re.match(r'^\d{2}/\d{2}/\d{4}$', fecha):
                                            continue
                                        # -------------------------

                                        extracted_records.append({
                                            'Fecha': fecha,
                                            'Origen': cols[1].get_text(strip=True),
                                            'Precio Mín': cols[2].get_text(strip=True),
                                            'Precio Max': cols[3].get_text(strip=True),
                                            'Precio Frec': cols[4].get_text(strip=True),
                                            'Producto_Buscado': producto_nombre,
                                            'Año': anio,
                                            'Mes': mes,
                                            'Semana': semana
                                        })

                                if extracted_records:
                                    df_limpio = pd.DataFrame(extracted_records)
                                    datos_completos.append(df_limpio)
                                    print(f"  [+] Extraído perfecto: Semana {semana}")

                    except Exception as e:
                        print(f"  [!] Error de ejecución: {e}")

    if datos_completos:
        df_final = pd.concat(datos_completos, ignore_index=True)
        df_final.to_csv('PRUEBA_precios_sniim.csv', index=False, encoding='utf-8-sig')
        print(f"\n¡Prueba exitosa! Se extrajeron {len(df_final)} filas limpias.")
        return df_final
    else:
        print("\nPrueba finalizada sin extracción.")
        return pd.DataFrame()

# Recuerda cerrar el archivo en tu Data Wrangler/Excel antes de correr esto
df_prueba = prueba_rapida_sniim_estricta()

Iniciando MODO PRUEBA ESTRICto...

  [+] Extraído perfecto: Semana 1
  [+] Extraído perfecto: Semana 2
  [+] Extraído perfecto: Semana 3

¡Prueba exitosa! Se extrajeron 3 filas limpias.
